# Task 3, Step 5: Adaptive Sampling

**Manual step 5.** Reveal rollouts in the saved orders and check stopping after $n\in\{4,8,16\}$, otherwise stop at 32. Both controllers return the current self-consistency answer, so the comparison isolates *stopping*.

- **Agreement:** a unique, non-`NO_ANSWER` leading answer has share at least $\tau$.
- **Quality-aware:** agreement, plus self-consistency and RWC winners agree, plus RWC margin at least $\gamma$. Scores are self-certainty for MATH and VisConf for MathVista, recomputed from revealed candidates only.

Settings are chosen on the 200 development questions: lowest mean token cost within one percentage point of fixed-32 self-consistency. They are frozen before held-out evaluation. These are offline simulations of stopping between completed rollouts.

In [ ]:
import os, sys, json
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "task3_test_time_compute")):
    root = os.path.dirname(root)
sys.path.insert(0, root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from task3_test_time_compute.src.notebook_utils import *
setup_notebook()

In [ ]:
from task3_test_time_compute.src.adaptive_stopping import select_policies, evaluate_policy, TAUS, GAMMAS
from task3_test_time_compute.src.evaluate import make_orders
grids, selections, outcomes, tests = {}, {}, {}, {}
for dataset in DATASETS:
    dev, test, _ = split_questions(dataset)
    tests[dataset] = test
    grids[dataset], selections[dataset] = select_policies(dev, make_orders(dev))
    grids[dataset].to_csv(f"{RESULTS_DIR}/adaptive_dev_grid_{dataset}.csv", index=False)
    print(DATASETS[dataset], "selected on development:", selections[dataset])
with open(f"{RESULTS_DIR}/adaptive_selected_settings.json", "w") as f:
    json.dump(selections, f, indent=1)

## 5.1 Development grid
Cells inside the one-point accuracy constraint are outlined. The selected setting is the cheapest outlined cell.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7.2))
for row, (dataset, label) in enumerate(DATASETS.items()):
    grid = grids[dataset][grids[dataset]["family"] == "quality_aware"]
    fixed = grids[dataset][grids[dataset]["family"] == "fixed_32"].iloc[0]
    cost = grid.pivot(index="gamma", columns="tau", values="mean_cost") / fixed["mean_cost"] * 100
    accuracy = grid.pivot(index="gamma", columns="tau", values="accuracy") * 100
    admissible = grid.pivot(index="gamma", columns="tau", values="admissible")
    sns.heatmap(cost, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[row, 0], cbar_kws={"label": "% of fixed-32 tokens"})
    sns.heatmap(accuracy, annot=True, fmt=".1f", cmap="viridis", ax=axes[row, 1], cbar_kws={"label": "Accuracy (%)"})
    for i in range(admissible.shape[0]):
        for j in range(admissible.shape[1]):
            if admissible.iloc[i, j]:
                for ax in axes[row]:
                    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False, edgecolor="white", linewidth=2.5))
    axes[row, 0].set(title=f"{label}: token cost", xlabel=r"$\tau$", ylabel=r"$\gamma$")
    axes[row, 1].set(title=f"{label}: dev accuracy (fixed-32: {fixed['accuracy'] * 100:.1f}%)", xlabel=r"$\tau$", ylabel=r"$\gamma$")
save_figure("task3_5_dev_grid")

## 5.2 Held-out results
The settings above are now frozen. This is the only place held-out questions meet the controllers.

In [ ]:
rows = []
for dataset, label in DATASETS.items():
    orders = make_orders(tests[dataset])
    for name, setting in [("Fixed-32", {"tau": None, "gamma": None}), ("Agreement", selections[dataset]["agreement"]), ("Quality-aware", selections[dataset]["quality_aware"])]:
        outcome = evaluate_policy(tests[dataset], orders, setting["tau"], setting["gamma"])
        outcomes[(dataset, name)] = outcome
        outcome.assign(dataset=dataset, policy=name).to_csv(f"{RESULTS_DIR}/adaptive_outcomes_{dataset}_{name.lower().replace('-', '_')}.csv", index=False)
        per_q = outcome.groupby("sample_id")[["correct", "cost", "rollouts"]].mean()
        rows.append({"Dataset": label, "Policy": name, "tau": setting["tau"], "gamma": setting["gamma"], "Accuracy (%)": per_q["correct"].mean() * 100, "Mean tokens": per_q["cost"].mean(), "Mean rollouts": per_q["rollouts"].mean()})
heldout = pd.DataFrame(rows)
heldout.to_csv(f"{RESULTS_DIR}/adaptive_heldout.csv", index=False)
display(heldout.round(2))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
offline = {d: pd.read_csv(f"{RESULTS_DIR}/offline_{d}.csv") for d in DATASETS}
for ax, (dataset, label) in zip(axes[:2], DATASETS.items()):
    curve = offline[dataset][offline[dataset]["method"] == "self_consistency"].groupby("N")[["accuracy", "token_cost"]].mean()
    ax.plot(curve["token_cost"], curve["accuracy"] * 100, marker="o", color="gray", label="Fixed-N self-consistency")
    for n, row in curve.iterrows():
        ax.annotate(f"N={n}", (row["token_cost"], row["accuracy"] * 100), textcoords="offset points", xytext=(4, -10), fontsize=8)
    for name, marker, color in [("Agreement", "s", "#3498db"), ("Quality-aware", "D", "#e74c3c")]:
        point = heldout[(heldout["Dataset"] == label) & (heldout["Policy"] == name)].iloc[0]
        ax.scatter(point["Mean tokens"], point["Accuracy (%)"], marker=marker, s=110, color=color, edgecolor="black", zorder=5, label=name)
    ax.set(title=label, xlabel="Mean tokens per question (log scale)", ylabel="Accuracy (%)", xscale="log"); ax.legend(fontsize=9)

stops = []
for (dataset, name), outcome in outcomes.items():
    if name != "Fixed-32":
        stops.append(outcome["rollouts"].value_counts(normalize=True).mul(100).rename("share").reset_index().assign(policy=f"{dataset}: {name}"))
stops = pd.concat(stops)
stops.to_csv(f"{RESULTS_DIR}/adaptive_stopping_distribution.csv", index=False)
stops.pivot(index="policy", columns="rollouts", values="share").fillna(0).plot(kind="barh", stacked=True, ax=axes[2], colormap="viridis", edgecolor="black")
axes[2].set(title="Where each policy stopped", xlabel="% of question-order pairs", ylabel=""); axes[2].legend(title="Rollouts", fontsize=8)
save_figure("task3_5_heldout")

## 5.3 Incorrect early stops
An early stop is **harmful** when it returns a wrong answer and the same order at fixed 32 would have returned the right one. Gold answers appear only in this retrospective analysis.

In [ ]:
rows = []
for dataset, label in DATASETS.items():
    fixed = outcomes[(dataset, "Fixed-32")].set_index(["sample_id", "order"])["correct"]
    for name in ["Agreement", "Quality-aware"]:
        outcome = outcomes[(dataset, name)].set_index(["sample_id", "order"])
        early = outcome[outcome["rollouts"] < 32]
        harmed = (early["correct"] == 0) & (fixed.loc[early.index] == 1)
        helped = (early["correct"] == 1) & (fixed.loc[early.index] == 0)
        rows.append({"Dataset": label, "Policy": name, "Stopped early (%)": 100 * len(early) / len(outcome), "Early stops wrong (%)": 100 * (early["correct"] == 0).mean() if len(early) else 0,
                     "Harmed vs. fixed-32 (%)": 100 * harmed.mean() if len(early) else 0, "Helped vs. fixed-32 (%)": 100 * helped.mean() if len(early) else 0})
early_table = pd.DataFrame(rows)
early_table.to_csv(f"{RESULTS_DIR}/adaptive_early_stop_analysis.csv", index=False)
display(early_table.round(2))